# Colab Retrain Mapan (Tambah Kelas 'Bukan Padi')
Jalankan cell di bawah ini satu per satu. Pastikan Runtime sudah menggunakan GPU (T4 GPU).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Karena Anda meletakkan file di root Google Drive (MyDrive)
BASE_DIR = '/content/drive/MyDrive'

# Ekstrak dataset_split_small.zip (jika Anda mengupload versi zip agar cepat)
!unzip -q -o "{BASE_DIR}/dataset_split_small.zip" -d "{BASE_DIR}/"

# Rename folder hasil ekstrak agar sesuai dengan nama aslinya
!rm -rf "{BASE_DIR}/dataset_split"
!mv "{BASE_DIR}/dataset_split_small" "{BASE_DIR}/dataset_split"
print("Dataset berhasil diekstrak dan disiapkan!")

In [ ]:
!pip install bing-image-downloader

import os
import random
import shutil
from bing_image_downloader import downloader

print("Membuat direktori Bukan Padi...")
for split in ['train', 'val']:
    os.makedirs(f"{BASE_DIR}/dataset_split/{split}/Bukan Padi", exist_ok=True)

queries = [
    "human face selfie close up",
    "people walking on street",
    "car parking lot",
    "indoor living room",
    "landscape scenery nature",
    "random everyday objects"
]

print("Mulai mengunduh gambar acak dari internet...")
for q in queries:
    downloader.download(q, limit=250, output_dir='raw_background', adult_filter_off=False, force_replace=False, timeout=60, verbose=False)

all_images = []
for root, dirs, files in os.walk('raw_background'):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            all_images.append(os.path.join(root, file))

random.shuffle(all_images)
print(f"Total gambar Bukan Padi terkumpul: {len(all_images)}")

max_images = min(1500, len(all_images))
train_count = int(max_images * 0.9)

train_images = all_images[:train_count]
val_images = all_images[train_count:max_images]

print(f"Menyalin {len(train_images)} gambar ke folder Latih (Train)...")
for img in train_images:
    shutil.copy(img, f"{BASE_DIR}/dataset_split/train/Bukan Padi/")
    
print(f"Menyalin {len(val_images)} gambar ke folder Validasi (Val)...")
for img in val_images:
    shutil.copy(img, f"{BASE_DIR}/dataset_split/val/Bukan Padi/")
    
print("Dataset 'Bukan Padi' berhasil ditambahkan!")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(rescale=1./127.5, offset=-1.0, horizontal_flip=True, rotation_range=20)
val_datagen = ImageDataGenerator(rescale=1./127.5, offset=-1.0)

train_gen = train_datagen.flow_from_directory(
    f"{BASE_DIR}/dataset_split/train", target_size=(224, 224), batch_size=32, class_mode='categorical'
)
val_gen = val_datagen.flow_from_directory(
    f"{BASE_DIR}/dataset_split/val", target_size=(224, 224), batch_size=32, class_mode='categorical'
)

NUM_CLASSES = len(train_gen.class_indices)
print(f"Total Kelas Sekarang: {NUM_CLASSES}")

model_path = f"{BASE_DIR}/model/rice_disease_model.h5"

# Fix kompatibilitas Keras 3 untuk model Keras 2 lama
from tensorflow.keras.layers import DepthwiseConv2D
class CustomDepthwiseConv2D(DepthwiseConv2D):
    def __init__(self, **kwargs):
        kwargs.pop('groups', None)
        super().__init__(**kwargs)

old_model = load_model(model_path, compile=False, custom_objects={'DepthwiseConv2D': CustomDepthwiseConv2D})

second_to_last_layer = old_model.layers[-2].output 
new_output = Dense(NUM_CLASSES, activation='softmax', name='new_predictions')(second_to_last_layer)

new_model = Model(inputs=old_model.input, outputs=new_output)

for layer in new_model.layers[:-10]:
    layer.trainable = False
for layer in new_model.layers[-10:]:
    layer.trainable = True

new_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(new_model.summary())

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

epochs = 5
callbacks = [EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True)]

history = new_model.fit(
    train_gen,
    epochs=epochs,
    validation_data=val_gen,
    callbacks=callbacks
)

new_model.save(f"{BASE_DIR}/model/rice_disease_model_v2.h5")
print("Model V2 berhasil disimpan!")

In [ ]:
!pip install tf2onnx

import tf2onnx
import json
import os
from pathlib import Path

output_dir = f"{BASE_DIR}/model/onnx_model"
os.makedirs(output_dir, exist_ok=True)
onnx_path = f"{output_dir}/model.onnx"
metadata_path = f"{output_dir}/metadata.json"

print("Mengonversi Model Keras ke ONNX (Opset 13)...")
onnx_model, _ = tf2onnx.convert.from_keras(new_model, opset=13)

with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"Model ONNX berhasil disimpan di: {onnx_path}")

class_names = list(train_gen.class_indices.keys())
metadata = {"class_names": class_names}

with open(metadata_path, "w") as f:
    json.dump(metadata, f)

print(f"Metadata berhasil dibuat di: {metadata_path}")
print("SELESAI! Silakan download folder onnx_model dan pindahkan ke public/models/rice-disease di Laravel Anda.")